# Curation Receipts

Tells users-api what walden did with each curation tonight (oxjob #2007). Before this, users-api's status worker could only
guess by watching the live index, so "not applied yet", "ignored by policy", "target gone" and "applied but the API prints the
name differently" all read as pending, or as a wrong "applied" (oxjob #1998: 3.48M curation-bot RAS rows read applied while
walden ignores them on matcher strings, which froze the RAS status lane for 10 days).

Runs at the very end of the End 2 End, after `Delete_Works_from_Elasticsearch` and `Sync_RAS_to_Elasticsearch`, so a receipt
exists only for a night that passed Guardrails and reached Elasticsearch. Nothing depends on this task; it writes nothing the
works build reads (0 works re-stamped).

## What it writes

- `openalex.curations.curation_receipts`: one row per curation walden considered (v1: RAS, work-author claims and removals).
  `outcome` is `applied` | `outranked` | `ignored_by_policy` | `target_gone` | `unsupported` | `unexplained`; `reason` and
  `detail` say why (the winning curation, the work a merged work became); `effect_live` says whether the served data shows
  what the curation asked anyway (an ignored bot add the matcher also makes). `changed_run_date` moves only when outcome,
  reason, detail or effect_live change: it is the key users-api pulls by.
- `openalex.curations.curation_receipt_runs`: one row per run (counts by kind and outcome), written after the MERGE commits.
  users-api reads a run only once its row exists.

`unexplained` (walden serves something no rule explains) is a bug canary, 0 rows when this shipped. It is counted and
sampled below, never fatal.

## How each outcome is decided

Read from what the night built, never re-derived: the RAS mapping MV, the claim view `work_authors_curated` (and #2008's
`work_author_curations_resolved` for a curation's resolved work), #2006's `work_status`, and the byline in
`openalex_works_base`. Claims and removals are judged by **effect** (who the view serves at the seat) and the cause is then
attributed, so the receipt cannot drift from the view's precedence rules. Replayed against the status worker's own live
checks on 3 Nov 2026: 99.1-99.5% agreement per kind, every disagreement an observer blind spot (oxjob #2007 EXPLORE).

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.curations.curation_receipts (
  curation_id      STRING NOT NULL,
  kind             STRING NOT NULL,   -- ras | work_author_claim | work_author_remove
  outcome          STRING NOT NULL,   -- applied | outranked | ignored_by_policy | target_gone | unsupported | unexplained
  reason           STRING,
  detail           STRING,
  effect_live      BOOLEAN,
  served_value     STRING,            -- debugging only; changes here do not move changed_run_date
  changed_run_date DATE NOT NULL,     -- the run whose receipt last changed outcome/reason/detail/effect_live: users-api's delta key
  first_run_date   DATE NOT NULL,
  updated_at       TIMESTAMP
)
CLUSTER BY (curation_id)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
COMMENT 'oxjob #2007: what walden did with each users-api curation (apply receipt). Written by CurationReceipts at the end of the End 2 End; read by the users-api curation status worker.';

CREATE TABLE IF NOT EXISTS openalex.curations.curation_receipt_runs (
  run_date     DATE NOT NULL,
  job_run_id   STRING,
  published_at TIMESTAMP,
  receipts     BIGINT,
  changed      BIGINT,
  unexplained  BIGINT,
  counts       MAP<STRING, BIGINT>
)
COMMENT 'oxjob #2007: one row per End 2 End run whose curation receipts are complete. users-api pulls a run only after its row exists.';

## Tonight's receipts

Staged in full (about 32M rows), then merged so only changed rows are rewritten. Three parts:

- **RAS** (log rows from users-api through the read-only federation link, filtered in Postgres like SyncRasCurations cell 5,
  created before tonight's RAS sync; mojibake re-keyed the same way): string not in `affiliation_strings_lookup` ->
  target_gone; the curation bot's row on a string the matcher answers -> ignored_by_policy (oxjob #1386: those strings take
  `ras_curations_without_bot`); not the latest (created, id) for its (string, institution) among the curations that apply
  to the string -> outranked; else applied iff the MV serves what the action asks.
- **Claims**: profile deleted -> target_gone (the view skips them); resolved work deleted or unknown -> target_gone; no seat on
  the resolved work with that exact printed name -> name_not_on_byline; every such seat past the byline's end ->
  seat_not_on_byline (CreateWorkAuthorships binds by position); the view serves the claimed author at a seat whose byline
  prints the claimed name (same tokens) -> applied, or prints a name sharing a token -> applied with reason
  byline_name_changed, or a different name -> target_gone seat_moved (the author is served on someone else's name); the view
  serves someone else -> outranked by the claim it serves, by the removal of this author on this work, or (a claim carried
  over from a merged-away work, oxjob #2008) by the claim made on the work that names the seat's author. A carried-over
  claim's detail names the work it was carried to (`via merge into W...`), applied or not. effect_live: the view serves the
  claimed author somewhere on the work's byline, under any printed name (users-api then shows the claim applied).
- **Removals**: resolved work deleted or unknown -> target_gone; the view still serves the author on the work -> outranked
  by the claim that keeps them; else applied.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.curations.curation_receipts_tonight AS
WITH
-- ---------- RAS ----------
ras_cutoff AS (SELECT MAX(updated_datetime) AS t FROM openalex.institutions.ras_curations),
ras_log AS (
  SELECT c.id AS curation_id, c.user_id, c.entity_id AS s0, c.action, c.created,
         CAST(REGEXP_REPLACE(c.value, '^https?://openalex\\.org/I', '') AS BIGINT) AS institution_id
  FROM openalex_users.public.curations c
  WHERE c.entity = 'ras' AND c.property = 'institution_ids' AND c.action IN ('add', 'remove')
    AND c.value RLIKE '^https?://openalex\\.org/I[0-9]+$'
),
ras_gated AS (
  SELECT *, COALESCE(s0 RLIKE '[\\u00C2-\\u00DF\\u00E0-\\u00EF][\\u0080-\\u00BF\\u20AC\\u201A\\u0192\\u201E\\u2026\\u2020\\u2021\\u02C6\\u2030\\u0160\\u2039\\u0152\\u017D\\u2018\\u2019\\u201C\\u201D\\u2022\\u2013\\u2014\\u02DC\\u2122\\u0161\\u203A\\u0153\\u017E\\u0178]|[\\u0102\\u0139]', false) AS g
  FROM ras_log
  WHERE created < (SELECT t FROM ras_cutoff)
),
-- oxjob #801 re-key, same gate as SyncRasCurations; the UDF runs only on gated rows, in its own projection
ras_rk AS (
  SELECT curation_id, user_id, s0 AS s, action, created, institution_id FROM ras_gated WHERE NOT g
  UNION ALL
  SELECT curation_id, user_id, openalex.common.repair_mojibake(s0), action, created, institution_id FROM ras_gated WHERE g
),
ras_x AS (
  SELECT r.*,
    r.user_id IS NOT NULL AND r.user_id IN (SELECT id FROM openalex_users.public.users WHERE email = 'curation-bot@ourresearch.org') AS is_bot,
    ans.raw_affiliation_string IS NOT NULL AS is_matcher,
    asl.raw_affiliation_string IS NOT NULL AS in_lookup,
    mv.institution_ids AS served_ids
  FROM ras_rk r
  LEFT JOIN openalex.institutions.affiliation_strings_lookup asl ON asl.raw_affiliation_string = r.s
  LEFT JOIN openalex.institutions.affiliation_matcher_answers ans ON ans.raw_affiliation_string = r.s
  LEFT JOIN openalex.institutions.raw_affiliation_strings_institutions_mv mv ON mv.raw_affiliation_string = r.s
),
ras_w AS (
  SELECT x.*,
    -- the winner among the curations that apply to this string (people only on matcher strings)
    MAX_BY(curation_id, STRUCT(created, curation_id)) OVER (PARTITION BY s, institution_id, (is_matcher AND is_bot)) AS winner_id,
    CASE WHEN action = 'add' THEN COALESCE(ARRAY_CONTAINS(served_ids, institution_id), false)
         ELSE NOT COALESCE(ARRAY_CONTAINS(served_ids, institution_id), false) END AS effect_live
  FROM ras_x x
),
ras AS (
  SELECT curation_id, 'ras' AS kind,
    CASE WHEN NOT in_lookup THEN 'target_gone'
         WHEN is_matcher AND is_bot THEN 'ignored_by_policy'
         WHEN curation_id <> winner_id THEN 'outranked'
         WHEN effect_live THEN 'applied'
         ELSE 'unexplained' END AS outcome,
    CASE WHEN NOT in_lookup THEN 'string_not_in_lookup'
         WHEN is_matcher AND is_bot THEN 'bot_on_matcher_string'
         WHEN curation_id <> winner_id THEN 'by_curation' END AS reason,
    CASE WHEN in_lookup AND NOT (is_matcher AND is_bot) AND curation_id <> winner_id THEN winner_id END AS detail,
    effect_live,
    ARRAY_JOIN(TRANSFORM(SLICE(ARRAY_SORT(COALESCE(served_ids, array())), 1, 50), i -> CONCAT('I', i)), ' ') AS served_value
  FROM ras_w
),
-- ---------- work-author claims and removals ----------
-- One row per curation with the work it applies to tonight (#2008: a curation on a merged-away work follows the merge).
wa_cur AS (
  SELECT curation_id, kind, user_id, author_id, raw_author_name, created, work_id, resolved_work_id, via_merge
  FROM openalex.works.work_author_curations_resolved
),
ws AS (SELECT work_id, status, merged_into FROM openalex.works.work_status),
-- every seat on a curated (resolved) work, with the author the view serves and the name the byline prints there
seat AS (
  SELECT wa.work_id, wa.author_sequence, wa.raw_author_name AS seat_name, v.author_id AS served_author_id,
         wa.author_sequence < COALESCE(SIZE(b.authorships), 0) AS on_byline,
         GET(b.authorships, wa.author_sequence).raw_author_name AS byline_name
  FROM openalex.works.work_authors wa
  JOIN (SELECT DISTINCT resolved_work_id FROM wa_cur) w ON w.resolved_work_id = wa.work_id
  JOIN openalex.works.work_authors_curated v ON v.work_id = wa.work_id AND v.author_sequence = wa.author_sequence
  LEFT JOIN openalex.works.openalex_works_base b ON b.id = wa.work_id
),
seat_k AS (
  SELECT *, ARRAY_SORT(FILTER(SPLIT(REGEXP_REPLACE(LOWER(seat_name), '[^\\p{L}\\p{N}]+', ' '), ' '), t -> t <> '')) AS sk,
            ARRAY_SORT(FILTER(SPLIT(REGEXP_REPLACE(LOWER(byline_name), '[^\\p{L}\\p{N}]+', ' '), ' '), t -> t <> '')) AS bk
  FROM seat
),
claim_seats AS (
  SELECT c.curation_id,
    COUNT(s.work_id) AS n_seats,
    COUNT_IF(s.on_byline) AS n_on_byline,
    COUNT_IF(s.on_byline AND s.served_author_id = c.author_id AND (s.sk = s.bk)) AS n_ok,
    COUNT_IF(s.on_byline AND s.served_author_id = c.author_id AND NOT (s.sk = s.bk) AND SIZE(ARRAY_INTERSECT(s.sk, s.bk)) > 0) AS n_changed,
    COUNT_IF(s.on_byline AND s.served_author_id = c.author_id) AS n_served,
    MAX(CASE WHEN s.on_byline THEN s.served_author_id END) AS other_author_id,
    MAX(CASE WHEN s.on_byline AND s.served_author_id = c.author_id AND SIZE(ARRAY_INTERSECT(s.sk, s.bk)) = 0 THEN s.byline_name END) AS moved_to
  FROM wa_cur c
  LEFT JOIN seat_k s ON s.work_id = c.resolved_work_id AND s.seat_name = c.raw_author_name
  WHERE c.kind = 'claim'
  GROUP BY c.curation_id
),
-- authors the view serves anywhere on the byline of each curated work: a claim's effect is live when its author is served
-- on its work under any printed name (a claim carried to a merge winner that prints the name in another form, oxjob #2008)
served_on_work AS (
  SELECT DISTINCT work_id, served_author_id AS author_id FROM seat WHERE on_byline AND served_author_id IS NOT NULL
),
-- the claim the view serves on each (work, printed name), for attributing 'outranked'
served_claim AS (
  SELECT c.resolved_work_id, c.raw_author_name, c.author_id, MAX_BY(c.curation_id, STRUCT(c.created, c.curation_id)) AS curation_id
  FROM wa_cur c WHERE c.kind = 'claim' GROUP BY ALL
),
latest_removal AS (
  SELECT resolved_work_id, author_id, MAX_BY(curation_id, STRUCT(created, curation_id)) AS curation_id
  FROM wa_cur WHERE kind = 'remove' GROUP BY ALL
),
-- #2008: a carried-over claim never takes a seat from an author that a claim made on the work itself names
direct_claim AS (
  SELECT resolved_work_id, author_id, MAX_BY(curation_id, STRUCT(created, curation_id)) AS curation_id
  FROM wa_cur WHERE kind = 'claim' AND NOT via_merge GROUP BY ALL
),
claims AS (
  SELECT c.curation_id, 'work_author_claim' AS kind,
    CASE WHEN da.author_id IS NOT NULL THEN 'target_gone'
         WHEN ws.status IS NULL OR ws.status <> 'live' THEN 'target_gone'
         WHEN s.n_seats = 0 THEN 'target_gone'
         WHEN s.n_on_byline = 0 THEN 'target_gone'
         WHEN s.n_ok > 0 OR s.n_changed > 0 THEN 'applied'
         WHEN s.n_served > 0 THEN 'target_gone'
         WHEN sc.curation_id IS NOT NULL OR lr.curation_id IS NOT NULL OR dcl.curation_id IS NOT NULL THEN 'outranked'
         ELSE 'unexplained' END AS outcome,
    CASE WHEN da.author_id IS NOT NULL THEN 'profile_deleted'
         WHEN ws.status IS NULL THEN 'work_not_found'
         WHEN ws.status <> 'live' THEN CONCAT('work_', ws.status)
         WHEN s.n_seats = 0 THEN 'name_not_on_byline'
         WHEN s.n_on_byline = 0 THEN 'seat_not_on_byline'
         WHEN s.n_ok > 0 THEN NULL
         WHEN s.n_changed > 0 THEN 'byline_name_changed'
         WHEN s.n_served > 0 THEN 'seat_moved'
         WHEN sc.curation_id IS NOT NULL THEN 'by_claim'
         WHEN lr.curation_id IS NOT NULL THEN 'by_removal'
         WHEN dcl.curation_id IS NOT NULL THEN 'by_claim' END AS reason,
    CASE WHEN da.author_id IS NOT NULL THEN NULL
         WHEN ws.status IS NULL OR ws.status <> 'live' THEN NULL
         WHEN s.n_seats = 0 OR s.n_on_byline = 0 THEN CASE WHEN c.via_merge THEN CONCAT('via merge into W', c.resolved_work_id) END
         WHEN s.n_ok > 0 OR s.n_changed > 0 THEN CASE WHEN c.via_merge THEN CONCAT('via merge into W', c.resolved_work_id) END
         WHEN s.n_served > 0 THEN CONCAT('byline now prints: ', LEFT(s.moved_to, 200))
         WHEN sc.curation_id IS NOT NULL THEN sc.curation_id
         WHEN lr.curation_id IS NOT NULL THEN lr.curation_id
         WHEN dcl.curation_id IS NOT NULL THEN dcl.curation_id END AS detail,
    sw.author_id IS NOT NULL AS effect_live,
    CAST(s.other_author_id AS STRING) AS served_value
  FROM wa_cur c
  JOIN claim_seats s ON s.curation_id = c.curation_id
  LEFT JOIN (SELECT DISTINCT author_id FROM openalex.authors.deleted_authors) da ON da.author_id = c.author_id
  LEFT JOIN ws ON ws.work_id = c.resolved_work_id
  LEFT JOIN served_on_work sw ON sw.work_id = c.resolved_work_id AND sw.author_id = c.author_id
  LEFT JOIN served_claim sc
    ON sc.resolved_work_id = c.resolved_work_id AND sc.raw_author_name = c.raw_author_name
   AND sc.author_id = s.other_author_id AND sc.author_id <> c.author_id
  LEFT JOIN latest_removal lr ON lr.resolved_work_id = c.resolved_work_id AND lr.author_id = c.author_id
  LEFT JOIN direct_claim dcl
    ON c.via_merge AND dcl.resolved_work_id = c.resolved_work_id
   AND dcl.author_id = s.other_author_id AND dcl.author_id <> c.author_id
  WHERE c.kind = 'claim'
),
remove_seats AS (
  SELECT r.curation_id, COUNT(s.work_id) AS n_served
  FROM wa_cur r
  LEFT JOIN seat s ON s.work_id = r.resolved_work_id AND s.served_author_id = r.author_id AND s.on_byline
  WHERE r.kind = 'remove'
  GROUP BY r.curation_id
),
keeping_claim AS (
  SELECT resolved_work_id, author_id, MAX_BY(curation_id, STRUCT(created, curation_id)) AS curation_id
  FROM wa_cur WHERE kind = 'claim' GROUP BY ALL
),
removes AS (
  SELECT r.curation_id, 'work_author_remove' AS kind,
    CASE WHEN ws.status IS NULL OR ws.status <> 'live' THEN 'target_gone'
         WHEN s.n_served = 0 THEN 'applied'
         WHEN kc.curation_id IS NOT NULL THEN 'outranked'
         ELSE 'unexplained' END AS outcome,
    CASE WHEN ws.status IS NULL THEN 'work_not_found'
         WHEN ws.status <> 'live' THEN CONCAT('work_', ws.status)
         WHEN s.n_served = 0 THEN NULL
         WHEN kc.curation_id IS NOT NULL THEN 'by_claim' END AS reason,
    CASE WHEN ws.status = 'live' AND s.n_served = 0 AND r.via_merge THEN CONCAT('via merge into W', r.resolved_work_id)
         WHEN ws.status = 'live' AND s.n_served > 0 THEN kc.curation_id END AS detail,
    s.n_served = 0 AS effect_live,
    CAST(s.n_served AS STRING) AS served_value
  FROM wa_cur r
  JOIN remove_seats s ON s.curation_id = r.curation_id
  LEFT JOIN ws ON ws.work_id = r.resolved_work_id
  LEFT JOIN keeping_claim kc ON kc.resolved_work_id = r.resolved_work_id AND kc.author_id = r.author_id
  WHERE r.kind = 'remove'
)
SELECT * FROM ras
UNION ALL SELECT * FROM claims
UNION ALL SELECT * FROM removes;

## Merge into the receipt table

A row is rewritten only when its receipt changed; `changed_run_date` moves only when outcome, reason, detail or effect_live
change (a served-value change alone updates the row but is not news for users-api). Curations deleted in users-api drop out.

Guards: one receipt per curation id, and tonight's staged receipts must hold at least 90% of the table's rows (a broken federation read or an empty view would
otherwise delete most receipts). On failure the task fails alone: nothing downstream depends on it and the night's publish
already happened. Fix the input and re-run this task from the run page (repair run).

In [ ]:
%sql
SELECT CASE
  WHEN (SELECT COUNT(*) - COUNT(DISTINCT curation_id) FROM openalex.curations.curation_receipts_tonight) > 0
  THEN RAISE_ERROR(CONCAT('CURATION RECEIPTS GUARD: ',
    CAST((SELECT COUNT(*) - COUNT(DISTINCT curation_id) FROM openalex.curations.curation_receipts_tonight) AS STRING),
    ' curation ids have two receipts tonight (an input stopped being one row per key). Nothing was merged.'))
END;

SELECT CASE
  WHEN (SELECT COUNT(*) FROM openalex.curations.curation_receipts) > 0
   AND (SELECT COUNT(*) FROM openalex.curations.curation_receipts_tonight)
       < 0.9 * (SELECT COUNT(*) FROM openalex.curations.curation_receipts)
  THEN RAISE_ERROR(CONCAT('CURATION RECEIPTS GUARD: staged ',
    CAST((SELECT COUNT(*) FROM openalex.curations.curation_receipts_tonight) AS STRING), ' receipts vs ',
    CAST((SELECT COUNT(*) FROM openalex.curations.curation_receipts) AS STRING),
    ' in the table (< 90%). An input (federation, a view) is broken; fix it and repair-run this task. Nothing was merged.'))
END;

MERGE INTO openalex.curations.curation_receipts AS t
USING (SELECT * FROM openalex.curations.curation_receipts_tonight) AS s
ON t.curation_id = s.curation_id
WHEN MATCHED AND NOT (t.kind <=> s.kind AND t.outcome <=> s.outcome AND t.reason <=> s.reason
                      AND t.detail <=> s.detail AND t.effect_live <=> s.effect_live) THEN UPDATE SET
  t.kind = s.kind, t.outcome = s.outcome, t.reason = s.reason, t.detail = s.detail, t.effect_live = s.effect_live,
  t.served_value = s.served_value, t.changed_run_date = CAST(:run_date AS DATE), t.updated_at = CURRENT_TIMESTAMP()
WHEN MATCHED AND NOT (t.served_value <=> s.served_value) THEN UPDATE SET
  t.served_value = s.served_value, t.updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (curation_id, kind, outcome, reason, detail, effect_live, served_value, changed_run_date, first_run_date, updated_at)
  VALUES (s.curation_id, s.kind, s.outcome, s.reason, s.detail, s.effect_live, s.served_value,
          CAST(:run_date AS DATE), CAST(:run_date AS DATE), CURRENT_TIMESTAMP())
WHEN NOT MATCHED BY SOURCE THEN DELETE;

-- users-api pages this table by curation_id (ORDER BY curation_id LIMIT 50000). A MERGE does not cluster what it
-- writes, so cluster it here; otherwise every page scans the whole table.
OPTIMIZE openalex.curations.curation_receipts;

## Record the run

Written last: users-api pulls a run's changed receipts only once this row exists, so it never reads a half-merged night.

In [ ]:
%sql
DELETE FROM openalex.curations.curation_receipt_runs WHERE run_date = CAST(:run_date AS DATE);

INSERT INTO openalex.curations.curation_receipt_runs
SELECT CAST(:run_date AS DATE), :job_run_id, CURRENT_TIMESTAMP(),
  (SELECT COUNT(*) FROM openalex.curations.curation_receipts),
  (SELECT COUNT(*) FROM openalex.curations.curation_receipts WHERE changed_run_date = CAST(:run_date AS DATE)),
  (SELECT COUNT(*) FROM openalex.curations.curation_receipts WHERE outcome = 'unexplained'),
  (SELECT MAP_FROM_ENTRIES(COLLECT_LIST(STRUCT(k, n))) FROM (
     SELECT CONCAT(kind, ':', outcome) AS k, COUNT(*) AS n FROM openalex.curations.curation_receipts GROUP BY ALL));

SELECT * FROM openalex.curations.curation_receipt_runs ORDER BY run_date DESC LIMIT 3;

## Verify

In [ ]:
%sql
-- What changed tonight, and a sample of anything unexplained (a bug canary: walden serves something no rule explains).
SELECT kind, outcome, reason, COUNT(*) AS n, COUNT_IF(changed_run_date = CAST(:run_date AS DATE)) AS changed_tonight
FROM openalex.curations.curation_receipts GROUP BY ALL ORDER BY kind, n DESC;

In [ ]:
%sql
SELECT * FROM openalex.curations.curation_receipts WHERE outcome = 'unexplained' ORDER BY updated_at DESC LIMIT 50;